# Solution to p12

Because receiver index is the row and sender index is the column, direct batched multiplication `A @ X` sums sender features for every receiver. Dividing by the sender-axis row degree gives the requested mean.

In [ ]:
import torch

node_features = torch.tensor([
    [[1.,0.,0.],[0.,2.,0.],[0.,0.,3.],[1.,1.,1.]],
    [[2.,0.,0.],[0.,4.,0.],[0.,0.,6.],[2.,2.,2.]],
], dtype=torch.float32)
adjacency = torch.tensor([
    [[1,1,0,0],[0,1,1,0],[1,0,1,1],[0,0,1,1]],
    [[1,0,1,0],[1,1,0,0],[0,1,1,1],[1,0,0,1]],
], dtype=torch.uint8)
ATOL = 1e-6
RTOL = 1e-6

def mean_neighbor_aggregate(node_features, adjacency):
    if node_features.ndim != 3 or adjacency.ndim != 3:
        raise ValueError("inputs must be batched rank-3 tensors")
    B, N, D = node_features.shape
    if adjacency.shape != (B,N,N):
        raise ValueError("adjacency shape mismatch")
    if node_features.dtype != torch.float32 or adjacency.dtype != torch.uint8:
        raise ValueError("dtype mismatch")
    if node_features.device.type != "cpu" or adjacency.device.type != "cpu":
        raise ValueError("inputs must be on CPU")
    if not bool(torch.all((adjacency == 0) | (adjacency == 1))):
        raise ValueError("adjacency must be binary")
    adjacency_float = adjacency.to(torch.float32)
    degree = adjacency_float.sum(dim=2, keepdim=True)
    if bool(torch.any(degree == 0)):
        raise ValueError("every receiver must have positive degree")
    messages = torch.bmm(adjacency_float, node_features) / degree
    return messages

messages = mean_neighbor_aggregate(node_features, adjacency)
degree = adjacency.to(torch.float32).sum(dim=2, keepdim=True)


### Answer check

In [ ]:
expected = torch.tensor([
 [[.5,1,0],[0,1,1.5],[2/3,1/3,4/3],[.5,.5,2]],
 [[1,0,3],[1,2,0],[2/3,2,8/3],[2,1,1]],
], dtype=torch.float32)
assert messages.shape == (2,4,3) and degree.shape == (2,4,1)
torch.testing.assert_close(messages, expected, atol=ATOL, rtol=RTOL)
torch.testing.assert_close(degree.squeeze(-1), adjacency.sum(dim=2).float(), atol=0, rtol=0)
